<a href="https://colab.research.google.com/github/Sanskar-CB/movie-recommendation-system/blob/main/Movies_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPOSITORY/blob/main/YOUR_NOTEBOOK.ipynb)


In [1]:
# Install required libraries
!pip install -q pandas numpy scikit-learn surprise matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

print("✅ Libraries successfully imported!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.3 MB/s eta 0:00:00
✅ Libraries successfully imported!


In [2]:
# Set seed for reproducibility
np.random.seed(42)

# Sample Movie Dataset
movies_data = {
    'movie_id': range(1, 11),
    'title': [
        'The Dark Knight', 'Inception', 'Interstellar', 'The Matrix',
        'Pulp Fiction', 'Fight Club', 'Forrest Gump', 'The Godfather',
        'Avatar', 'Titanic'
    ],
    'genres': [
        'Action Crime Drama', 'Action Sci-Fi Thriller', 'Adventure Drama Sci-Fi',
        'Action Sci-Fi', 'Crime Drama', 'Drama Thriller',
        'Drama Romance', 'Crime Drama', 'Action Adventure Sci-Fi', 'Drama Romance'
    ],
    'description': [
        'Batman fights organized crime in Gotham with the help of Jim Gordon.',
        'A thief steals corporate secrets through dream-sharing technology.',
        'A team of explorers travel through a wormhole in space to save humanity.',
        'A computer hacker learns about the true nature of his reality.',
        'The lives of two mob hitmen, a boxer, and a gangster intertwine.',
        'An unfulfilled office worker forms an underground fight club.',
        'The presidencies of Kennedy and Johnson unfold through the perspective of an Alabama man.',
        'The aging patriarch of an organized crime dynasty transfers control to his reluctant son.',
        'A paraplegic Marine dispatched to the moon Pandora on a unique mission.',
        'A seventeen-year-old aristocrat falls in love with a kind but poor artist aboard the Titanic.'
    ]
}

movies_df = pd.DataFrame(movies_data)

# Combine genres and descriptions for content-based feature extraction
movies_df['metadata'] = movies_df['genres'] + ' ' + movies_df['description']

print(f"Movies Dataset Loaded: {movies_df.shape[0]} movies")
movies_df[['movie_id', 'title', 'genres']].head()

Movies Dataset Loaded: 10 movies


,movie_id,title,genres
0,1,The Dark Knight,Action Crime Drama
1,2,Inception,Action Sci-Fi Thriller
2,3,Interstellar,Adventure Drama Sci-Fi
3,4,The Matrix,Action Sci-Fi
4,5,Pulp Fiction,Crime Drama


In [3]:
# Create TF-IDF Vectorizer to convert metadata into numerical vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_df['metadata'])

# Compute Cosine Similarity matrix between all movies
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Helper function to get recommendations based on movie title
def recommend_movies_content(title, cosine_sim=cosine_sim, df=movies_df, top_n=3):
    # Find movie index
    idx_list = df.index[df['title'].str.lower() == title.lower()].tolist()
    if not idx_list:
        print(f"Movie '{title}' not found in database.")
        return
    idx = idx_list[0]

    # Get pairwise similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get scores of top_n most similar movies (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]

    print(f"🎥 Recommendations for '<strong>{df.loc[idx, 'title']}</strong>':")
    recommendations = df.iloc[movie_indices][['title', 'genres']].copy()
    recommendations['similarity_score'] = [round(i[1], 4) for i in sim_scores]
    return recommendations

# Test Content-Based Recommendations
recommend_movies_content('Inception', top_n=3)

🎥 Recommendations for '<strong>Inception</strong>':


,title,genres,similarity_score
3,The Matrix,Action Sci-Fi,0.1614
8,Avatar,Action Adventure Sci-Fi,0.1452
2,Interstellar,Adventure Drama Sci-Fi,0.0979


In [4]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Generate Synthetic User Ratings Data
n_ratings = 500
users = np.random.randint(1, 51, size=n_ratings)  # 50 users
movie_ids = np.random.randint(1, 11, size=n_ratings) # 10 movies
ratings = np.random.choice([1, 2, 3, 4, 5], size=n_ratings, p=[0.1, 0.1, 0.2, 0.3, 0.3])

ratings_df = pd.DataFrame({
    'userID': users,
    'movie_id': movie_ids,
    'rating': ratings
}).drop_duplicates(subset=['userID', 'movie_id'])

# Prepare data for Surprise library
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['userID', 'movie_id', 'rating']], reader)

# Train-Test Split
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD Model
svd_model = SVD(n_factors=20, random_state=42)
svd_model.fit(trainset)

# Evaluate model on test set
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"✅ SVD Model Trained! Test RMSE: {rmse:.4f}")

RMSE: 1.1909
✅ SVD Model Trained! Test RMSE: 1.1909


In [5]:
def recommend_movies_hybrid(user_id, top_n=3):
    """
    Predicts top movies for a specific user using collaborative filtering SVD score.
    """
    all_movie_ids = movies_df['movie_id'].unique()

    # Find movies already rated by the user
    rated_movies = ratings_df[ratings_df['userID'] == user_id]['movie_id'].tolist()
    unrated_movies = [m for m in all_movie_ids if m not in rated_movies]

    # Predict ratings for unrated movies
    predictions = []
    for m_id in unrated_movies:
        pred = svd_model.predict(user_id, m_id)
        predictions.append((m_id, pred.est))

    # Sort predictions by highest estimated rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_predictions = predictions[:top_n]

    top_movie_ids = [p[0] for p in top_predictions]
    result = movies_df[movies_df['movie_id'].isin(top_movie_ids)][['movie_id', 'title', 'genres']].copy()
    result['predicted_rating'] = [round(p[1], 2) for p in top_predictions]

    print(f"\n🌟 Top Recommendations for User {user_id}:")
    return result

# Test Hybrid / Collaborative Recommendation for User 5
recommend_movies_hybrid(user_id=5, top_n=3)


🌟 Top Recommendations for User 5:


,movie_id,title,genres,predicted_rating
1,2,Inception,Action Sci-Fi Thriller,3.68
4,5,Pulp Fiction,Crime Drama,3.56
9,10,Titanic,Drama Romance,3.55
